# 05 - Handoff between tiles

**Goal:** move the mover across a tile edge with no central controller.

* Coils fade in/out smoothly by distance (`handoff.fade_inner_mm` / `fade_outer_mm`, Jansen 2008).
* The tile under the mover *owns* it and computes currents for nearby coils, including the
  neighbour's; ownership moves once the mover is `hysteresis_mm` past the edge.
* Neighbour tiles apply the currents they are sent `link_delay_steps` control steps late.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from maglev_sim import *
from maglev_sim import plotting

In [ ]:
cfg = with_changes(load_config("../configs/baseline.yaml"), {"grid.rows": 1, "grid.cols": 2})
s = build_system(cfg)
plotting.plot_layout(s, pose=s.pose(x=0.100, y=0.050));

## 1. Quasi-static check: currents along a line across the edge (x = 100 mm)

In [ ]:
from maglev_sim.handoff import straight_line_sweep
df = straight_line_sweep(s, [0.05, 0.05], [0.15, 0.05], n=81)
plotting.plot_handoff(df, s);

The pattern repeats every coil pitch and nothing special happens at the tile edge - the
"seamless" goal. The sharp swings are the coils commutating under the magnets, made worse by
this layout's weak tilt (torque) authority - a design point worth improving.

## 2. Full closed loop across the edge

In [ ]:
p0, p1 = np.r_[0.05, 0.05, s.gap], np.r_[0.15, 0.05, s.gap]
res = simulate(s, Trajectory.move(p0, p1, 0.2, 0.8), duration=1.3)
print(res.summary())
print("handoff log (step, kind, from tile, to tile):", res.info["handoff_log"])
plotting.plot_simulation(res);

## 3. How much link latency can we tolerate?

In [ ]:
rows = []
for delay in [0, 1, 2, 5, 10]:
    c = with_changes(cfg, {"handoff.link_delay_steps": delay})
    r = simulate(build_system(c), Trajectory.move(p0, p1, 0.2, 0.8), duration=1.3)
    rows.append({"delay_steps": delay, "delay_ms": delay * 1000 / c["control"]["rate_Hz"],
                 **{k: r.info[k] for k in ["rms_error_mm", "max_error_mm", "peak_current_A", "touchdown"]}})
import pandas as pd
pd.DataFrame(rows).round(3)

This directly answers the lit-review question "can the tile link carry handoff data at the
control rate?" - read off the largest delay that keeps tracking acceptable, then pick UART/CAN/...
accordingly. To try a different ownership/communication strategy, subclass `TileNetwork` in
`src/maglev_sim/tiles.py`.